In [1]:
# conda install transformers datasets torch

In [4]:
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import torch

# 1. Load and prepare the dataset
dataset = load_dataset("csv", data_files="data.csv", cache_dir=None)

# Split the dataset into train and validation sets
train_dataset = dataset["train"]

# 2. Load the T5 model and tokenizer
model_name = "t5-small"  # You can use t5-base or t5-large for more capacity
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

# 3. Preprocess the dataset
def preprocess_function(examples):
    # Format input as "question: <question> context: <context>"
    inputs = [f"question: {q} context: {c}" for q, c in zip(examples['question'], examples['context'])]
    targets = examples['answer']
    
    # Tokenize inputs and targets
    model_inputs = tokenizer(inputs, max_length=512, padding=True, truncation=True)
    labels = tokenizer(targets, max_length=128, padding=True, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply preprocessing
tokenized_dataset = train_dataset.map(preprocess_function, batched=True, keep_in_memory=True)


# 4. Set up training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=10_000,
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=500,
    evaluation_strategy="no",  # Disable evaluation
)


# 5. Initialize the Trainer
trainer = Trainer(
    model=model,                         # the model to be trained
    args=training_args,                  # training arguments
    train_dataset=tokenized_dataset,     # training dataset
    tokenizer=tokenizer,                 # tokenizer for text processing
)

# 6. Train the model
trainer.train()

# 7. Save the fine-tuned model
model.save_pretrained('./fine_tuned_t5')
tokenizer.save_pretrained('./fine_tuned_t5')

# 8. Optionally, evaluate the model
# Split your dataset into training and evaluation
eval_dataset = tokenized_dataset.select(range(len(tokenized_dataset) // 2))  # Example split

# Evaluate using the eval_dataset
results = trainer.evaluate(eval_dataset=eval_dataset)

# Print evaluation results
print(results)



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Map: 100%|██████████| 4/4 [00:00<00:00, 666.71 examples/s]
C:\Users\kavin\AppData\Roaming\Python\Python310\site-packages\transformers\training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
100%|██████████| 3/3 [00:02<00:00,  1.08it/s]


{'train_runtime': 2.7837, 'train_samples_per_second': 4.311, 'train_steps_per_second': 1.078, 'train_loss': 3.4494810104370117, 'epoch': 3.0}


100%|██████████| 1/1 [00:00<00:00, 142.80it/s]

{'eval_loss': 2.4117627143859863, 'eval_runtime': 0.2281, 'eval_samples_per_second': 8.77, 'eval_steps_per_second': 4.385, 'epoch': 3.0}


In [1]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load the fine-tuned model and tokenizer
model_path = './fine_tuned_t5'
model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)



C:\Users\kavin\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
# Define the question and context
question = "what supervised learninng??"
context = "Supervised learning is a type of machine learning where the model is trained using labeled data. The goal is to teach the model to make predictions based on known outcomes."

# Format input
input_text = f"question: {question} context: {context}"

# Tokenize the input
input_ids = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).input_ids

# Generate the output
output_ids = model.generate(input_ids, max_length=250, num_beams=5, early_stopping=True)

# Decode the output to text
answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"Answer: {answer}")


Answer: machine learning
